In [ ]:
%%capture
!pip install contractions emoji

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os, re, contractions

df = pd.read_json('data/Dataset for Detection of Cyber-Trolls.json', lines= True)
df.head()

,content,annotation,extras
0,Get fucking real dude.,"{'notes': '', 'label': ['1']}",NaN
1,She is as dirty as they come and that crook ...,"{'notes': '', 'label': ['1']}",NaN
2,why did you fuck it up. I could do it all day...,"{'notes': '', 'label': ['1']}",NaN
3,Dude they dont finish enclosing the fucking s...,"{'notes': '', 'label': ['1']}",NaN
4,WTF are you talking about Men? No men thats n...,"{'notes': '', 'label': ['1']}",NaN


In [ ]:
#count of number of classes
df.annotation.value_counts()

,count
annotation,
"{'notes': '', 'label': ['0']}",12179
"{'notes': '', 'label': ['1']}",7822


In [ ]:
def preprocessing(sample):
    review = re.sub('[^a-zA-Z\d]',' ',sample)       #Removing annotations
    review = review.lower()                                 #Converting everything to lower case
    review = review.split()                                 #Splitting each word in a review into a separate list
    review = ' '.join(review)                               #Joining all the words into a single list
    return review

In [ ]:
df['content'] = df['content'].apply(preprocessing)
df['content'].head()

,content
0,get fucking real dude
1,she is as dirty as they come and that crook re...
2,why did you fuck it up i could do it all day t...
3,dude they dont finish enclosing the fucking sh...
4,wtf are you talking about men no men thats not...


In [ ]:
# Function to expand contractions
def expand_contractions(text):
    return contractions.fix(text)

# Apply the function to the 'content' column
df['content'] = df['content'].apply(expand_contractions)

df['content'].head()

,content
0,get fucking real dude
1,she is as dirty as they come and that crook re...
2,why did you fuck it up i could do it all day t...
3,dude they do not finish enclosing the fucking ...
4,wtf are you talking about men no men that is n...


- check for emoji

In [ ]:
import pandas as pd
import emoji

def contains_emoji(text):
    # Use the emoji package to identify emojis in the text
    emojis = [c for c in text if c in emoji.EMOJI_DATA]
    return emojis

# text data
df['contains_emoji'] = df['content'].apply(contains_emoji)

In [ ]:
df['contains_emoji'].value_counts()

,count
contains_emoji,
[],20001


# Embedding Generation: Hybrid


## Fasttext and Bert

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp310-cp310-linux_x86_64.whl size=4296181 sha256=ecff848b3ed9fbbc555b0dd6f13106719d1e967859a3b25c4fbeef289ea0bbd8
  Stored in directory: /root/.cache/pip/wheels/0d/a2/00/81db54d3e6a8199b829d58e02cec2ddb20ce3e59fad8d3c92a
Successfully built fasttext


In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertModel
import tensorflow as tf
import numpy as np
import fasttext

In [ ]:
df = df.head(10)

In [ ]:
import fasttext.util
fasttext.util.download_model('en', if_exists='ignore')

In [ ]:
# Load FastText pre-trained embeddings
model_path = '/kaggle/working/cc.en.300.bin'
fasttext_model = fasttext.load_model(model_path)

# Load the BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Preprocess the text data
def preprocess_text(text):
    # You can customize this function based on your specific preprocessing needs.
    # For example, lowercasing, tokenization, and removing punctuation.
    text = text.lower()
    tokens = tokenizer.tokenize(text)
    return ' '.join(tokens)

df['cleaned_content'] = df['content'].apply(preprocess_text)

# Extract BERT embeddings
def extract_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Average pooling of BERT embeddings
    return embeddings.detach().numpy()

df['bert_embeddings'] = df['cleaned_content'].apply(extract_bert_embeddings)

In [ ]:
df.head()

## hybrid concatenation

In [ ]:
def load_fasttext_embeddings(text, model_path):
    word_embeddings = []
    text = text['cleaned_content']
    for word in text.split():
        try:
            word_vector = fasttext_model.get_word_vector(word)
            word_embeddings.append(word_vector)
        except KeyError:
            # Handle words not found in the FastText model
            pass

    if word_embeddings:
        return np.mean(word_embeddings, axis=0)
    else:
        # If no FastText embeddings were found, return zeros
        return np.zeros(fasttext_model.get_dimension())

# Combine pre-trained FastText and BERT embeddings
def combine_embeddings(text):
    fasttext_embeddings = load_fasttext_embeddings(text, fasttext_model)
    bert_embeddings = text['bert_embeddings']

    # Check dimensions and pad if necessary
    if len(fasttext_embeddings.shape) == 1:
        fasttext_embeddings = np.expand_dims(fasttext_embeddings, axis=0)

    if len(bert_embeddings.shape) == 1:
        bert_embeddings = np.expand_dims(bert_embeddings, axis=0)

    return np.concatenate((fasttext_embeddings, bert_embeddings), axis=1)

df['hybrid_embeddings'] = df.apply(combine_embeddings, axis=1)

# Convert the embeddings to TensorFlow tensors
hybrid_embeddings = np.stack(df['hybrid_embeddings'].values)
hybrid_embeddings = tf.convert_to_tensor(hybrid_embeddings, dtype=tf.float32)


In [ ]:
import numpy as np

# Convert TensorFlow tensor to NumPy array
hybrid_embeddings_np = hybrid_embeddings.numpy()

# Save NumPy array to a file using np.save
np.save('saved_files/hybrid_embedding_array.npy', hybrid_embeddings_np)